# Wörterbuchprüfung mit Levenshtein-Distanz

In dieser Übung implementiert ihr Korrekturvorschläge, baut ein Wörterbuch aus Textdateien auf und korrigiert damit einen neuen Text.

## Arbeitsauftrag

Ergänze die folgenden Funktionen und führe jeweils die zugehörige Testzelle aus.

- Abschnitt 2 (Korrekturvorschläge) – `suggest_corrections`
- Abschnitt 3 (Wörterbuch aus Textdateien) – `build_vocabulary`


## 1 Distanz

Die Implementierung der Levenshtein-Distanz aus dem ersten Notebook ist hier bereits vorgegeben.

In [ ]:
def initialize_matrix(source: str, target: str) -> list[list[int]]:

    D = []
    for _ in range(len(source) + 1):
        D.append([0] * (len(target) + 1))
    
    for i in range(len(source) + 1):
        D[i][0] = i

    for j in range(len(target) + 1):
        D[0][j] = j

    return D


def calculate_cell(D: list[list[int]], source: str, target: str, i: int, j: int) -> int: 

    if i < 1 or j < 1: 
        raise ValueError('i and j must be at least 1')
    
    deletion = D[i - 1][j] + 1
    insertion = D[i][j - 1] + 1

    substitution_cost = 0 if source[i - 1] == target[j - 1] else 1
    diagonal = D[i - 1][j - 1] + substitution_cost

    return min(deletion, insertion, diagonal)


def levenshtein_matrix(source: str, target: str) -> list[list[int]]:
    D = initialize_matrix(source, target)

    for i in range(1, len(source) + 1):
        for j in range(1, len(target) + 1):
            D[i][j] = calculate_cell(D, source, target, i, j)

    return D


def levenshtein_distance(source: str, target: str) -> int:
    D = levenshtein_matrix(source, target)
    return D[-1][-1]
    

In [ ]:
print(levenshtein_distance('cat', 'coast'))


## 2 Ähnlichste Wörter finden und Korrekturen vorschlagen

Implementiere `suggest_corrections`:

- Die Funktion erhält ein Eingabewort und eine Menge von Wörterbuchwörtern.
- Ist das Eingabewort im Wörterbuch, brauchen wir keinen Korrekturvorschlag und die Funktion soll eine leere Liste zurückgeben.
- Ist das Eingabewort nicht im Wörterbuch, sollen **alle** Wörter des Wörterbuchs mit minimaler Levenshtein-Distanz zum Eingabewort zurückgegeben werden.
- Die Distanz sollte maximal 2 Editieroperationen betragen. Gibt es im Wörterbuch kein Wort mit einer Distanz $<= 2$, soll die Funktion eine leere Liste zurückgeben.


In [ ]:
def suggest_corrections(word: str, vocabulary: set[str]) -> list[str]:
    raise NotImplementedError('Implement suggest_corrections')

In [ ]:
import unittest
check = unittest.TestCase()

check.assertEqual(suggest_corrections('cat', {'cat', 'dog'}), [])
check.assertEqual(suggest_corrections('cab', {'cat', 'car', 'dog'}), ['car', 'cat'])
check.assertEqual(suggest_corrections('cab', {'caterpillar', 'dogs', 'house'}), [])

check.assertEqual(suggest_corrections('cat', set()), [])
check.assertEqual(suggest_corrections('cater', {'cat'}), ['cat'])
check.assertEqual(suggest_corrections('caterp', {'cat'}), [])

print('Correction tests passed.')


Diese Korrekturlogik hat Limitationen: 
- Ein korrektes Wort kann im Wörterbuch fehlen.
- Umgekehrt bleibt ein Fehler unentdeckt, wenn das falsch verwendete Wort selbst im Wörterbuch steht.
- Bei mehreren gleich nahen Wörtern ist die Korrektur nicht eindeutig.

## 3 Wörterbuch aus Textdateien aufbauen

Ihr sollt aus den drei Textdateien in `data/training` ein Wörterbuch aufbauen. 
Jede Zeile enthält einen Satz.
`tokenize` zerlegt eine Zeile in Folgen englischer Buchstaben und wandelt sie in Kleinbuchstaben um.
Satzzeichen, Zahlen und Umlaute zählen nicht zu diesen Wörtern.


In [ ]:
from pathlib import Path
import re


def tokenize(text: str) -> list[str]:
    return re.findall(r'[a-z]+', text.lower())


print(tokenize('# Recycled glass, recycled panels, 100 degrees'))


Implementiere `build_vocabulary`. Öffne die Textdateien im übergebenen Ordner mit UTF-8, lies sie zeilenweise und füge die Wörter jeder Zeile einer Menge hinzu.

In [ ]:
def build_vocabulary(directory: Path) -> set[str]:
    raise NotImplementedError('Implement build_vocabulary')

In [ ]:
file_vocabulary = build_vocabulary(Path('data/training'))

check.assertTrue({'brickley', 'fibres', 'cabbage'} <= file_vocabulary)
check.assertIn('mabel', file_vocabulary)
check.assertNotIn('insulaton', file_vocabulary)

print('Vocabulary tests passed.')
print(f'Vocabulary size: {len(file_vocabulary)}')


## 4 Eine Textdatei korrigieren

Wir prüfen die Wörter einer Zeile mit unserem Wörterbuch. Für ein unbekanntes Wort wählen wir zufällig einen der nächsten Vorschläge. Gibt es keinen Vorschlag, bleibt das Wort unverändert.

`correct_word` korrigiert ein einzelnes Wort. `correct_line` zerlegt die Zeile mit `tokenize` in Wörter, korrigiert diese und verbindet sie mit Leerzeichen. Dabei gehen Großschreibung, Satzzeichen und Zahlen verloren.

In [ ]:
import random


def correct_word(word: str, vocabulary: set[str]) -> str:
    suggestions = suggest_corrections(word, vocabulary)
    if not suggestions:
        return word

    return random.choice(suggestions)


def correct_line(line: str, vocabulary: set[str]) -> str:
    corrected_words = []

    for word in tokenize(line):
        corrected_words.append(correct_word(word, vocabulary))

    return ' '.join(corrected_words)


In [ ]:
check.assertEqual(correct_line('Insulaton, glass!\n', {'insulation', 'glass'}), 'insulation glass')
check.assertIn(correct_line('plne', {'pane', 'plane'}), {'pane', 'plane'})
check.assertEqual(correct_line('extraordinary!', {'cat'}), 'extraordinary')
check.assertEqual(correct_line('cat  42!\n', {'cat'}), 'cat')
check.assertEqual(correct_line('', {'cat'}), '')

print('Line correction tests passed.')


Die Funktion `correct_file` liest die angegebene Datei und schreibt den korrigierten Text in den Ausgabeordner. Sie gibt den Pfad zur Ergebnisdatei zurück.

In [ ]:
def correct_file(input_path: Path, output_directory: Path, vocabulary: set[str]) -> Path:
    
    output_directory.mkdir(parents=True, exist_ok=True)
    output_path = output_directory / input_path.name

    with input_path.open(encoding='utf-8') as source:
        with output_path.open('w', encoding='utf-8') as target:
            for line in source:
                target.write(correct_line(line, vocabulary) + '\n')

    return output_path


In [ ]:
input_path = Path('data/to_correct/01.txt')
output_path = correct_file(input_path, Path('data/corrected'), file_vocabulary)
print(f'Corrected file: {output_path}')

with output_path.open(encoding='utf-8') as file:
    for line_number, line in enumerate(file):
        if line_number == 5:
            break
        print(line, end='')
        

Die Eingabedatei ist eine fehlerhafte Fassung von `data/training/01.txt`. Vergleicht die korrigierte Datei mit dem Original. Welche Fehler wurden richtig korrigiert? Wo wurde ein anderes Wort eingesetzt?